# Fashionpedia · Mask R-CNN — Entrenamiento en Google Colab

Este notebook entrena un modelo **Mask R-CNN** (ResNet50-FPN v2) sobre el dataset **Fashionpedia**, usando la GPU de Google Colab.

**Contenido:**
1. Comprobación de GPU e instalación de dependencias
2. Montaje de Google Drive (dataset + guardado de checkpoints)
3. Definición del `Dataset`, del `modelo` y del bucle de entrenamiento
4. Entrenamiento y guardado del modelo
5. Visualización de resultados de segmentación

> **Antes de ejecutar:** en Colab ve a `Entorno de ejecución → Cambiar tipo de entorno de ejecución` y selecciona **GPU** (T4 o superior).

## 1. Comprobar GPU e instalar dependencias

In [ ]:
!nvidia-smi

In [ ]:
# pycocotools no viene preinstalado en Colab
!pip install -q pycocotools

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Montar Google Drive

Se usa para leer el dataset (imágenes + anotaciones) y para guardar los checkpoints del modelo entrenado, de forma que no se pierdan si se reinicia el entorno de Colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configuración de rutas

Ajusta estas rutas según dónde tengas subido el dataset de Fashionpedia dentro de tu Google Drive. Por defecto se asume una estructura como:

```
MyDrive/
└── fashionpedia/
    ├── train/                                  (imágenes de entrenamiento)
    ├── test_no_humans/                         (imágenes de validación)
    ├── instances_train_no_humans.json
    └── instances_test_no_humans.json
```

Los checkpoints se guardarán en `checkpoint_dir`, también dentro de Drive.

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/fashionpedia"

TRAIN_IMAGES_DIR = f"{DRIVE_ROOT}/train_no_humans"
TRAIN_ANNOTATIONS = f"{DRIVE_ROOT}/instances_train_no_humans.json"

VAL_IMAGES_DIR = f"{DRIVE_ROOT}/test_no_humans"
VAL_ANNOTATIONS = f"{DRIVE_ROOT}/instances_test_no_humans.json"

CHECKPOINT_DIR = f"{DRIVE_ROOT}/checkpoints"

# Hiperparámetros de entrenamiento
IMAGE_SIZE = 256
BATCH_SIZE = 2
EPOCHS = 10
LEARNING_RATE = 0.005
WORKERS = 2          # en Colab, 2 workers suele ir bien
MAX_TRAIN_IMAGES = None   # pon un número (p.ej. 500) para pruebas rápidas
MAX_VAL_IMAGES = None
PRETRAINED = True

## 4. Dataset (`FashionpediaDataset`)

Mismo código que `dataset.py`, adaptado para vivir dentro del notebook.

In [ ]:
import os
import random
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset
from pycocotools.coco import COCO
from pycocotools import mask as coco_mask


class FashionpediaDataset(Dataset):
    """
      __len__: cuántos elementos tiene el dataset en total
      __getitem__: obtiene el elemento número i (imagen + máscara)

    PyTorch llama a __getitem__ repetidamente durante el entrenamiento,
    así que aquí solo definimos cómo se construye un solo par
    (imagen, máscara) a partir de un índice.
    """

    def __init__(self, images_dir, annotations_file, image_size=256):
        self.images_dir = images_dir
        self.image_size = image_size

        self.coco = COCO(annotations_file)

        all_images_ids = list(sorted(self.coco.imgs.keys()))

        self.image_ids = [
            img_id for img_id in all_images_ids
            if os.path.isfile(
                os.path.join(self.images_dir, self.coco.loadImgs(img_id)[0]["file_name"])
            )
        ]

        n_total = len(all_images_ids)
        n_encontradas = len(self.image_ids)
        if n_encontradas < n_total:
            print(
                f"Aviso: de {n_total} imágenes en el JSON, solo se encontraron "
                f"{n_encontradas} en '{self.images_dir}'. Se usarán solo esas."
            )

        category_ids = sorted(self.coco.getCatIds())
        self.categoryid_to_index = {category_id: i + 1 for i, category_id in enumerate(category_ids)}
        self.index_to_categoryid = {v: k for k, v in self.categoryid_to_index.items()}

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        img_info = self.coco.loadImgs(image_id)[0]

        img_path = os.path.join(self.images_dir, img_info["file_name"])
        image = Image.open(img_path).convert("RGB")
        image_np = np.array(image)

        target = self._build_target(img_info, image_id)

        image_np = np.array(Image.fromarray(image_np).resize((self.image_size, self.image_size), Image.BILINEAR))

        image_tensor = torch.from_numpy(image_np).permute(2, 0, 1).float() / 255.0
        scale_x = self.image_size / img_info["width"]
        scale_y = self.image_size / img_info["height"]
        target["boxes"][:, [0, 2]] *= scale_x
        target["boxes"][:, [1, 3]] *= scale_y
        target["masks"] = torch.stack([
            torch.from_numpy(
                np.array(
                    Image.fromarray(instance_mask).resize(
                        (self.image_size, self.image_size), Image.NEAREST
                    )
                )
            )
            for instance_mask in target.pop("_masks")
        ]).to(torch.uint8) if target["boxes"].shape[0] else torch.zeros(
            (0, self.image_size, self.image_size), dtype=torch.uint8
        )
        target["area"] = target["masks"].flatten(1).sum(dim=1).to(torch.float32)

        return image_tensor, target

    def _build_target(self, img_info, image_id):
        height, width = img_info["height"], img_info["width"]
        annotation_ids = self.coco.getAnnIds(imgIds=image_id)
        annotations = self.coco.loadAnns(annotation_ids)

        masks = []
        labels = []
        boxes = []
        areas = []

        for annotation in annotations:
            segmentation = annotation["segmentation"]
            if isinstance(segmentation, list):
                rles = coco_mask.frPyObjects(segmentation, height, width)
                rle = coco_mask.merge(rles)
            elif isinstance(segmentation["counts"], list):
                rle = coco_mask.frPyObjects(segmentation, height, width)
            else:
                rle = segmentation

            instance_mask = coco_mask.decode(rle)
            if instance_mask.ndim == 3:
                instance_mask = np.any(instance_mask, axis=2)
            instance_mask = instance_mask.astype(np.uint8)
            rows, columns = np.where(instance_mask)
            if rows.size == 0:
                continue

            masks.append(instance_mask)
            labels.append(self.categoryid_to_index[annotation["category_id"]])
            boxes.append([columns.min(), rows.min(), columns.max() + 1, rows.max() + 1])
            areas.append(float(instance_mask.sum()))

        return {
            "boxes": torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4),
            "labels": torch.tensor(labels, dtype=torch.int64),
            "masks": torch.empty((0, height, width), dtype=torch.uint8),
            "image_id": torch.tensor([image_id], dtype=torch.int64),
            "area": torch.tensor(areas, dtype=torch.float32),
            "iscrowd": torch.zeros((len(masks),), dtype=torch.int64),
            "_masks": masks,
        }

## 5. Modelo (`get_model`, `get_device`, `load_model`)

Mismo código que `model.py`.

In [ ]:
from torchvision.models.detection import (
    maskrcnn_resnet50_fpn_v2,
    MaskRCNN_ResNet50_FPN_V2_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

# Fashionpedia tiene 46 categorías + 1 clase de background
NUM_CLASSES = 47


def get_device():
    if torch.cuda.is_available():
        device = torch.device("cuda:0")
        print(f"Usando GPU: {torch.cuda.get_device_name(0)}")
    else:
        device = torch.device("cpu")
        print("No se ha detectado GPU. Usando CPU.")
    return device


def get_model(num_classes=NUM_CLASSES, pretrained=True):
    weights = MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT if pretrained else None
    model = maskrcnn_resnet50_fpn_v2(weights=weights)

    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, hidden_layer, num_classes)

    return model


def load_model(checkpoint_path, num_classes=NUM_CLASSES, device="cpu"):
    """Carga un modelo previamente entrenado desde un checkpoint .pth"""
    model = get_model(num_classes, pretrained=False)
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
    else:
        model.load_state_dict(checkpoint)

    return model

## 6. Bucle de entrenamiento

Mismo código que `train.py`, sin `argparse` (los hiperparámetros ya se definieron arriba).

In [ ]:
import time
from typing import Dict, List, Tuple
from torch.utils.data import DataLoader, Subset


def collate_fn(batch: List[Tuple[torch.Tensor, torch.Tensor]]):
    images, targets = zip(*batch)
    return list(images), list(targets)


def build_dataset(dataset, limit):
    if limit is None or limit >= len(dataset):
        return dataset
    return Subset(dataset, range(limit))


def move_targets_to_device(targets: List[Dict[str, torch.Tensor]], device: torch.device):
    return [{key: value.to(device) for key, value in target.items()} for target in targets]


def train_one_epoch(model, loader, optimizer, device, epoch: int, log_every: int = 10) -> float:
    model.train()
    total_loss = 0.0

    for step, (images, targets) in enumerate(loader, start=1):
        images = [image.to(device) for image in images]
        targets = move_targets_to_device(targets, device)

        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        if step % log_every == 0 or step == len(loader):
            print(f"Época {epoch} | paso {step}/{len(loader)} | pérdida: {loss.item():.4f}")

    return total_loss / max(len(loader), 1)


@torch.no_grad()
def evaluate_loss(model, loader, device) -> float:
    # Mask R-CNN solo devuelve pérdidas en modo train; no se calculan gradientes.
    model.train()
    total_loss = 0.0

    for images, targets in loader:
        images = [image.to(device) for image in images]
        targets = move_targets_to_device(targets, device)
        loss_dict = model(images, targets)
        total_loss += sum(loss_dict.values()).item()

    return total_loss / max(len(loader), 1)


def train_model(
    model,
    train_dataset,
    val_dataset,
    device,
    *,
    epochs,
    batch_size,
    learning_rate,
    workers=0,
    checkpoint_dir="checkpoints",
    max_train_images=None,
    max_val_images=None,
):
    train_dataset = build_dataset(train_dataset, max_train_images)
    val_dataset = build_dataset(val_dataset, max_val_images)

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=workers, collate_fn=collate_fn,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        num_workers=workers, collate_fn=collate_fn,
    )

    optimizer = torch.optim.SGD(
        model.parameters(), lr=learning_rate, momentum=0.9, weight_decay=0.0005,
    )
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

    os.makedirs(checkpoint_dir, exist_ok=True)
    best_val_loss = float("inf")
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, device, epoch)
        val_loss = evaluate_loss(model, val_loader, device)
        scheduler.step()
        dt = time.time() - t0
        print(f"Época {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f} ({dt/60:.1f} min)")

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        checkpoint = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": val_loss,
        }
        torch.save(checkpoint, os.path.join(checkpoint_dir, "last.pth"))
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(checkpoint, os.path.join(checkpoint_dir, "best.pth"))
            print(f"  → Nuevo mejor modelo guardado (val_loss={val_loss:.4f})")

    return history

## 7. Crear los datasets y el modelo

In [ ]:
device = get_device()

train_dataset = FashionpediaDataset(
    TRAIN_IMAGES_DIR, TRAIN_ANNOTATIONS, image_size=IMAGE_SIZE,
)
val_dataset = FashionpediaDataset(
    VAL_IMAGES_DIR, VAL_ANNOTATIONS, image_size=IMAGE_SIZE,
)

print(f"Imágenes de entrenamiento: {len(train_dataset)}")
print(f"Imágenes de validación:    {len(val_dataset)}")

In [ ]:
model = get_model(num_classes=NUM_CLASSES, pretrained=PRETRAINED)
model.to(device)
print("Modelo creado y movido a:", device)

## 8. Entrenar

Esto puede tardar bastante según el número de imágenes y épocas. Los checkpoints (`last.pth` y `best.pth`) se guardan automáticamente en `CHECKPOINT_DIR` (dentro de Drive) tras cada época.

In [ ]:
history = train_model(
    model,
    train_dataset,
    val_dataset,
    device,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    workers=WORKERS,
    checkpoint_dir=CHECKPOINT_DIR,
    max_train_images=MAX_TRAIN_IMAGES,
    max_val_images=MAX_VAL_IMAGES,
)

### Curva de pérdida

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot(history["train_loss"], label="train_loss")
plt.plot(history["val_loss"], label="val_loss")
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.title("Evolución de la pérdida durante el entrenamiento")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 9. Guardar el modelo

El mejor checkpoint ya se guarda automáticamente en Drive durante el entrenamiento (`CHECKPOINT_DIR/best.pth`). Esta celda solo lo confirma.

In [ ]:
import os

best_ckpt = os.path.join(CHECKPOINT_DIR, "best.pth")
last_ckpt = os.path.join(CHECKPOINT_DIR, "last.pth")

print("¿Existe best.pth?", os.path.isfile(best_ckpt))
print("¿Existe last.pth?", os.path.isfile(last_ckpt))
print("Ruta:", CHECKPOINT_DIR)

## 10. Visualizar resultados de segmentación

Carga el mejor checkpoint y muestra las predicciones (cajas + máscaras) sobre varias imágenes de validación, junto a las anotaciones reales (ground truth).

In [ ]:
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np

SCORE_THRESHOLD = 0.5

# Recarga el mejor modelo entrenado
inference_model = load_model(best_ckpt, num_classes=NUM_CLASSES, device=device)
inference_model.to(device)
inference_model.eval()

# Nombres de categorías (id COCO -> nombre) para las etiquetas del gráfico
catid_to_name = {c["id"]: c["name"] for c in val_dataset.coco.loadCats(val_dataset.coco.getCatIds())}


def plot_prediction(image_tensor, target, prediction, ax_gt, ax_pred, score_threshold=0.5):
    image_np = image_tensor.permute(1, 2, 0).cpu().numpy()

    # --- Ground truth ---
    ax_gt.imshow(image_np)
    ax_gt.set_title("Ground truth")
    ax_gt.axis("off")
    for box, label in zip(target["boxes"], target["labels"]):
        x1, y1, x2, y2 = box.tolist()
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=1.5, edgecolor="lime", facecolor="none")
        ax_gt.add_patch(rect)
        category_id = val_dataset.index_to_categoryid.get(int(label), None)
        name = catid_to_name.get(category_id, str(int(label)))
        ax_gt.text(x1, max(y1 - 4, 0), name, color="lime", fontsize=7, backgroundcolor="black")

    # --- Predicción ---
    ax_pred.imshow(image_np)
    ax_pred.set_title("Predicción del modelo")
    ax_pred.axis("off")

    mask_overlay = np.zeros((*image_np.shape[:2], 4))
    colors = plt.cm.get_cmap("tab20", 20)

    keep = prediction["scores"] >= score_threshold
    boxes = prediction["boxes"][keep].cpu()
    labels = prediction["labels"][keep].cpu()
    scores = prediction["scores"][keep].cpu()
    masks = prediction["masks"][keep].cpu()

    for i, (box, label, score, mask) in enumerate(zip(boxes, labels, scores, masks)):
        color = colors(i % 20)
        m = (mask[0].numpy() > 0.5)
        for c in range(3):
            mask_overlay[..., c] = np.where(m, color[c], mask_overlay[..., c])
        mask_overlay[..., 3] = np.where(m, 0.5, mask_overlay[..., 3])

        x1, y1, x2, y2 = box.tolist()
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=1.5, edgecolor=color, facecolor="none")
        ax_pred.add_patch(rect)
        category_id = val_dataset.index_to_categoryid.get(int(label), None)
        name = catid_to_name.get(category_id, str(int(label)))
        ax_pred.text(x1, max(y1 - 4, 0), f"{name} {score:.2f}", color="white", fontsize=7, backgroundcolor=color)

    ax_pred.imshow(mask_overlay)


N_EXAMPLES = 4
indices = random.sample(range(len(val_dataset)), k=min(N_EXAMPLES, len(val_dataset)))

fig, axes = plt.subplots(len(indices), 2, figsize=(10, 5 * len(indices)))
if len(indices) == 1:
    axes = axes.reshape(1, 2)

with torch.no_grad():
    for row, idx in enumerate(indices):
        image_tensor, target = val_dataset[idx]
        prediction = inference_model([image_tensor.to(device)])[0]
        plot_prediction(image_tensor, target, prediction, axes[row, 0], axes[row, 1], SCORE_THRESHOLD)

plt.tight_layout()
plt.show()

---
### Notas
- Si Colab se desconecta o se agota la sesión, puedes reanudar el entrenamiento cargando `CHECKPOINT_DIR/last.pth` con `load_model(...)` y continuando desde ahí (basta con cargar el `state_dict` en un modelo nuevo antes de volver a llamar a `train_model`).
- Ajusta `BATCH_SIZE` según la memoria de la GPU asignada (una T4 de Colab suele aguantar `batch_size=2` con `image_size=256` sin problemas).
- `SCORE_THRESHOLD` controla qué tan seguras deben estar las predicciones para mostrarse; bájalo si no ves ninguna detección.